# Agon: Eos, training intelligent agents (restart on the current code)

One goal: agents whose decisions you can look at and call intelligent, and a number that says so.

**Why a restart.** The world changed since the last runs: the observation grew to 64 channels (fullness of a cell, how at ease animals are, trees' height and leaf tone), corpses rot, cells have limited room, animals can be tamed, trees have hidden kinds. Old checkpoints and trained networks cannot be loaded by this code, and `experiment.py` refuses to mix code versions. Everything here is a new run in a new folder (`exp_<code version>_...`).

**What this notebook does**
1. Loads the code, runs the tests that matter, checks that a competent scripted band can live in this world at all.
2. Scores *untrained* agents and *random* agents with `scripts/decision_check.py`. This is the bar training must clear.
3. Trains the `caveman` design (given survival skills, world model, sleep-consolidated learning) with `experiment.py validate --only 0`. It resumes: rerun the cell after a disconnect.
4. Re-scores the agents from the latest checkpoint at any time, and shows a playback whose Inspect, Decisions and Ideas tabs hold their choices.

**What "intelligent" means here, and what it does not.** `decision_check.py` reads each awake agent's choice every tick and asks whether it matches a plain survival rule (drink or seek water when thirsty, eat or seek food when hungry, sleep or rest at night, flee when a predator is within 3 cells), then divides by what random agents do. Beating chance on all four, while living longer than untrained agents, is the minimum. It does not prove planning or culture: the playback, the cause-and-effect stories and the world model's Ideas tab are where you look for that.

**Time.** Not measured on this code yet. The earlier probe took about 35 h for four 100M-step runs on an A100, so expect roughly 9 h for the one run here; step 4 benchmarks the real speed first.

## 1. GPU check

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "No GPU: Runtime > Change runtime type > A100 GPU, then restart."

## 2. Load the code

Upload the new `lifesim_code.zip` (made from the commit you are on: `git archive --format=zip -o lifesim_code.zip HEAD`), or set `ZIP_FROM_DRIVE`.

In [ ]:
import glob, html, json, shutil, subprocess, sys, zipfile
ZIP_FROM_DRIVE = ""   # e.g. "/content/drive/MyDrive/agon/lifesim_code.zip"
if ZIP_FROM_DRIVE and os.path.exists(ZIP_FROM_DRIVE):
    zip_path = ZIP_FROM_DRIVE
else:
    from google.colab import files
    zip_path = next(iter(files.upload()))
shutil.rmtree("/content/lifesim", ignore_errors=True)
os.makedirs("/content/lifesim")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/lifesim")
if not os.path.exists("/content/lifesim/pyproject.toml"):
    inner = next(os.path.dirname(p) for p in glob.glob("/content/lifesim/*/pyproject.toml"))
    for name in os.listdir(inner):
        shutil.move(os.path.join(inner, name), "/content/lifesim")
os.chdir("/content/lifesim")
!pip install -q -U "flax>=0.12.10" "optax>=0.2.8" pytest
!pip install -q --no-deps -e .
import importlib.metadata as md
v = {p: md.version(p) for p in ("jax", "flax", "optax")}
print(v)
assert tuple(int(x) for x in v["flax"].split(".")[:3]) >= (0, 12, 10), "Flax too old: Runtime > Restart session, rerun this cell."
sys.path.insert(0, "/content/lifesim")
from lifesim.presets import PRESETS, code_version
from lifesim import env as E
PRESET = "society30"   # 32 x 32 at half-hour ticks: 30x less compute per game day than `society`
assert os.path.exists("scripts/decision_check.py"), "This zip predates decision_check.py: upload the new lifesim_code.zip."
print("code version", code_version(), "| observation channels", E.N_OBS_CHANNELS, "| preset", PRESET)
assert E.N_OBS_CHANNELS == 64, "Not the current code (expected 64 observation channels)."

## 3. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE = f"/content/drive/MyDrive/agon/runs/intel_{code_version()}_{PRESET}"
os.makedirs(BASE, exist_ok=True)
print("saved to", BASE)

## 4. Tests and viability

The new systems (corpses, room, trees, taming) have their own tests: 4 files, 19 tests, about 10 minutes here, mostly compiling (the slowest single test takes 1 to 1.5 minutes). The cell prints each test as it finishes. If one fails, stop: a failing world is not worth training in.

The rest of the suite (`test_fauna`, `test_living`, `test_env`, and the others) takes 30+ minutes and is not needed to start training; CI runs it. The optional cell after this one runs it if you want. Then the viability check: can a scripted band live in this world at all?


In [ ]:
!python -m pytest -v -x -p no:cacheprovider tests/test_taming.py tests/test_dead.py tests/test_room.py tests/test_trees.py

In [ ]:
# Optional, 30+ minutes: the rest of the suite. Skip it to start training.
RUN_FULL_SUITE = False
if RUN_FULL_SUITE:
    !python -m pytest -v -x -p no:cacheprovider tests/test_fauna.py tests/test_living.py tests/test_env.py

In [ ]:
!python scripts/viability.py --preset {PRESET} --knowledge own --days 160 --seeds 3

## 5. Benchmark

How many parallel worlds fit on this GPU, and how fast. The full update ran out of memory at 16 worlds on a Colab A100, so only 4 and 8 are tried.

In [ ]:
WORLDS = None   # set 4 or 8 to skip the benchmark
if WORLDS is None:
    out = subprocess.run([sys.executable, "scripts/benchmark.py", "--preset", PRESET, "--worlds", "4", "8",
                          "--updates", "3", "--target-agent-steps", "1e8"], capture_output=True, text=True)
    print(out.stdout[-2500:] or out.stderr[-2500:])
    rec = [l for l in out.stdout.splitlines() if l.startswith("RECOMMENDED")]
    WORLDS = int(rec[-1].split("--worlds")[1].split()[0]) if rec else 4
ROOT = f"{BASE}_w{WORLDS}"      # one folder per world count: a checkpoint is never resumed with another batch size
os.makedirs(ROOT, exist_ok=True)
print("WORLDS =", WORLDS, "| runs in", ROOT)

## 6. The bar: random and untrained agents

Three seeds of 10 days. Random agents show chance; untrained networks show what a fresh brain does before any learning. Save the output: training has to beat both.

In [ ]:
!python scripts/decision_check.py --days 10 --seeds 3 --out {ROOT}/decision_check_untrained.json

## 7. Train

The `caveman` design: given survival skills, a world model, sleep-consolidated learning, survival as the only reward; then a 400-day continuous world with playback pages. `--only 0` is this run alone. It resumes from its checkpoint: after a disconnect, rerun steps 2, 3 and 5 (set `WORLDS` by hand to skip the benchmark), then this cell.

In [ ]:
STEPS = 1e8
!python scripts/experiment.py validate --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {STEPS} --playback-every 4 --only 0

## 7b. Why they stop improving: the stall ablations

The first run's log (`claude/research/design-audit-learning.md`): deaths per sixth of training went 1,025 → 325 → 280 → 177 → 209 → 276, and policy entropy fell to 0.07 nats (of 4.4 possible) a sixth of the way in and stayed there. The agents learned to drink, eat and sleep early, then became near-deterministic and stopped getting better at the rest (huddling on winter nights rose 30% → 64% and stuck there; cold deaths fell 271 → 49 per sixth, then rose; fire never). Two things hold them there: a small fixed entropy bonus, and the retention loss that pulls each brain toward what it already does.

`--only 1` is `caveman_entropy_floor` (`--entropy-target 0.4`: the bonus grows under the floor, shrinks above), `--only 2` is `caveman_no_retention`. Same preset, steps and seed as step 7. Run one, then step 8 with `RUN` pointing at it; the stall panel below shows whether learning kept moving. (`--only 3` is a shared-trunk design, kept as a flag but not the direction.)

In [ ]:
ABLATION = 1    # 1 entropy floor, 2 no retention
!python scripts/experiment.py validate --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {STEPS} --playback-every 4 --only {ABLATION}
# then: RUN = f"{ROOT}/validate/caveman_entropy_floor"  (or caveman_no_retention) in step 8

## 8. Are they getting intelligent?

Run this any time, also while step 7 is still going in another session: it reads the latest checkpoint if training has not finished. Compare each rule's rate against the random column and the untrained column; the last column is trained over random. Alive at end should rise above untrained.

Do not run it while step 7 is training in the same session: two heavy jobs on one GPU run out of memory.
**Ideas.** The world model gives each agent imagined futures for a few candidate acts (Ideas tab). The agent acts on them by a learned weight, `trust`, that starts at 0: ideas change nothing until acting on them pays. So in the plots below, trust staying at 0 means the ideas are decoration; trust above 0 with falling idea errors means they are being used; negative means they mislead. `decision_check.py` does not score ideas.


In [ ]:
RUN = f"{ROOT}/validate/caveman"
!python scripts/decision_check.py --run {RUN} --days 10 --seeds 3 --out {ROOT}/decision_check_trained.json

In [ ]:
import matplotlib.pyplot as plt
hist = json.load(open(f"{RUN}/history.json"))["history"]
share = lambda r, k: r["deaths"].get(k, 0) / max(1, sum(r["deaths"].values()))
panels = [("Mean age at death (ticks)", lambda r: r["mean_age_at_death"]),
          ("Thirst's share of deaths", lambda r: share(r, "thirst")),
          ("Starvation's share of deaths", lambda r: share(r, "starvation")),
          ("Asleep at night", lambda r: r.get("asleep_share_night")),
          ("Asleep by day", lambda r: r.get("asleep_share_day")),
          ("Policy entropy", lambda r: r.get("entropy")),
          ("Births per window", lambda r: r.get("births")),
          ("Share of experience learned", lambda r: r.get("learned_share")),
          # ideas: the world model's errors should fall, and trust (starts at 0) should rise above 0:
          # an agent acts on its ideas only as far as doing so has paid off; negative = they mislead
          ("Trust in ideas (0 = ignored)", lambda r: (r.get("world_model") or {}).get("trust_in_ideas")),
          ("Idea error: own body", lambda r: (r.get("world_model") or {}).get("own_state_error")),
          ("Idea error: what others do", lambda r: (r.get("world_model") or {}).get("belief_error")),
          ("Idea error: reward", lambda r: (r.get("world_model") or {}).get("reward_error"))]
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, (title, f) in zip(axes.flat, panels):
    ax.plot([r["agent_steps"] / 1e6 for r in hist], [float("nan") if f(r) is None else f(r) for r in hist])
    ax.set_title(title, fontsize=10); ax.set_xlabel("million agent-steps")
plt.tight_layout(); plt.show()

In [ ]:
# The stall panel: is learning still moving? Deaths by cause per sixth of training (fewer = better; read
# with births, since more mouths means more deaths), mean age at death, and policy entropy per sixth.
import collections
hist = json.load(open(f"{RUN}/history.json"))["history"]
K = 6; sz = max(1, len(hist) // K)
rows = []
for i in range(K):
    seg = hist[i * sz:(i + 1) * sz] if i < K - 1 else hist[i * sz:]
    d = collections.Counter()
    for r in seg: d.update(r["deaths"])
    ages = [r["mean_age_at_death"] for r in seg if r.get("mean_age_at_death") is not None]
    cold = collections.Counter()
    for r in seg: cold.update(r.get("cold_deaths", {}))
    rows.append((i, sum(d.values()), dict(d.most_common(4)), sum(r["births"] for r in seg),
                 sum(ages) / max(1, len(ages)), sum(r["entropy"] for r in seg) / len(seg), dict(cold)))
print(f"{'sixth':>5} {'deaths':>6} {'births':>6} {'age at death':>13} {'entropy':>8}  top causes")
for i, n, top, b, age, ent, cold in rows:
    print(f"{i:>5} {n:>6} {b:>6} {age:>13.0f} {ent:>8.3f}  {top}")
    if cold: print(f"{'':>5} cold: {cold}")
print("Stalled if deaths stop falling after the first sixth while entropy sits under ~0.1.")
print("Cold: 'open' deaths are a planning failure (a winter night in the open is damage, huddled or not);")
print("'sheltered' ones mean snow, a cold snap, or not resting; 'child' ones are the infants' own.")

## 9. Look at their decisions

A one-day playback of the trained agents. Click a person, then the Inspect tab: Decisions lists the acts it weighed with the policy's own probabilities and what it leaned on; Ideas shows what its world model imagined. Compare with an untrained page made the same way (`--decisions`).

In [ ]:
subprocess.run([sys.executable, "scripts/export_playback.py", "--run", RUN, "--days", "1", "--out", "/content/trained_decisions.html"], check=True)
subprocess.run([sys.executable, "scripts/export_playback.py", "--decisions", "--world-model", "--world", "living", "--slate", "caveman",
                "--agents", "8", "--grid", "32", "--days", "1", "--out", "/content/untrained_decisions.html"], check=True)
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{html.escape(open("/content/trained_decisions.html").read())}" style="width:100%;height:1100px;border:0"></iframe>'))

In [ ]:
# Download both pages to open in your browser
from google.colab import files
files.download("/content/trained_decisions.html"); files.download("/content/untrained_decisions.html")

### 9b. The decisions as text

The playback pages are about 28 MB each. This reads the decisions out of them as plain text: how confident the policy was, what agents actually chose when thirsty, hungry, cold, hurt or near a predator, and a spread of moments with the options weighed (with probabilities), what the choice leaned on, and the imagined plan with its expected effect on the body (Ideas). It is saved next to the run, a few KB, so it can be read from Drive. Pages are the continuous world's (every 4th chunk). For one agent: add `--agent 3`.

In [ ]:
CHUNKS = sorted(glob.glob(f"{RUN}/continuous/chunk_*.html"))
print([os.path.basename(c) for c in CHUNKS])
!python scripts/show_decisions.py {' '.join(CHUNKS)} --max 8 --out {RUN}/decisions_report.txt

## 10. Cause and effect and the full report

In [ ]:
!python scripts/causal_story.py {RUN}/continuous --max 40
!python scripts/experiment.py report --root {ROOT}
!python scripts/narrate.py {RUN}/continuous --out {RUN}/story.txt | tail -40   # the run as a story, from the record
